# 🧠 RAG-LLM Pipeline — Complete Walkthrough

This notebook provides a **step-by-step, interactive explanation** of every stage in the RAG pipeline.
Each cell demonstrates one component — you can run them in order to understand how queries flow through the system.

## Pipeline Overview

```
User Query
    │
    ├── 1. Query Enhancement (HyDE)
    ├── 2. Hybrid Search (Dense + Sparse + RRF)
    ├── 3. Cross-Encoder Reranking
    ├── 4. Prompt Optimization (Lost-in-Middle)
    ├── 5. LLM Generation
    └── 6. Self-RAG Reflection
         │
         ▼
    Final Answer + Sources
```

---

## 0. Setup & Imports

Before running this notebook, ensure:
- **Milvus** is running: `docker-compose up -d standalone`
- **LM Studio** is running on `http://127.0.0.1:1234`
- Dependencies are installed: `pip install -r requirements.txt`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Suppress tokenizer warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Path configured — ready to import RAG-LLM modules')

---

## 📥 Stage A: Document Ingestion Pipeline

Before we can query, we need to ingest documents. The ingestion pipeline has 3 stages:

1. **Load** — Read the file (PDF, TXT, MD)
2. **Chunk** — Split into overlapping text segments
3. **Embed & Store** — Convert to vectors and insert into Milvus

### A.1 Document Loading & Chunking

The `Chunker` class loads documents and splits them using a recursive character text splitter.
This is NOT simple paragraph splitting — it tries to split at natural boundaries (paragraphs → sentences → words).

In [ ]:
from src.services.ingestion.chunker import Chunker
from src.core.config import get_settings

settings = get_settings()

print(f'Chunk size:    {settings.chunk_size} characters')
print(f'Chunk overlap: {settings.chunk_overlap} characters')
print()

# Create a sample document
sample_text = """Retrieval-Augmented Generation (RAG) is a technique that combines 
information retrieval with text generation. When a user asks a question, the system 
first searches a knowledge base to find relevant documents, then provides those 
documents as context to a Large Language Model (LLM) to generate an informed answer.

The key advantage of RAG over pure LLM generation is that it grounds the model's 
responses in actual data, reducing hallucinations. The system can also cite its 
sources, making the answers more trustworthy and verifiable.

Modern RAG systems use vector databases to store document embeddings. When a query 
arrives, it's converted to a vector and compared against stored vectors using 
similarity search algorithms like cosine similarity or inner product."""

# Save to temp file and chunk
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(sample_text)
    temp_path = f.name

chunker = Chunker()
chunks = chunker.load_and_split(temp_path)

print(f'📄 Original text: {len(sample_text)} characters')
print(f'✂️  Split into: {len(chunks)} chunks')
print()
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk)} chars) ---')
    print(chunk[:200] + ('...' if len(chunk) > 200 else ''))
    print()

### A.2 Embedding — Converting Text to Vectors

Each chunk is converted to a **384-dimensional dense vector** using the `all-MiniLM-L6-v2` model.

The key insight: semantically similar texts produce vectors that are **close together** in vector space,
even if they use completely different words.

In [ ]:
from src.services.ingestion.embedder import get_embedder
import numpy as np

embedder = get_embedder()

# Embed a query and a document chunk
query = "What is RAG?"
chunk1 = "Retrieval-Augmented Generation combines retrieval with generation."
chunk2 = "The weather today is sunny and warm."

q_vec = embedder.embed_query(query)
c1_vec = embedder.embed_query(chunk1)
c2_vec = embedder.embed_query(chunk2)

# Cosine similarity
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f'📏 Embedding dimension: {len(q_vec)}')
print(f'📊 Model: {settings.embedding_model}')
print()
print(f'Query: "{query}"')
print(f'  ↔ Relevant chunk:   similarity = {cosine_sim(q_vec, c1_vec):.4f}  ✅ (high)')
print(f'  ↔ Irrelevant chunk: similarity = {cosine_sim(q_vec, c2_vec):.4f}  ❌ (low)')
print()
print('→ Higher similarity = more semantically related. This is how vector search finds relevant documents!')

### A.3 Storing in Milvus

Once embedded, chunks are stored in **Milvus** — a purpose-built vector database.

**Schema per chunk:**
| Field | Type | Description |
|-------|------|-------------|
| `id` | INT64 | Auto-generated primary key |
| `text` | VARCHAR | The original chunk text |
| `dense_embedding` | FLOAT_VECTOR[384] | The embedding vector |
| `source` | VARCHAR | Source filename |
| `user_id` | VARCHAR | Who uploaded it |
| `org_id` | VARCHAR | Organization |
| `doc_id` | VARCHAR | Parent document ID |
| `chunk_index` | INT64 | Position within document |
| `metadata` | JSON | Tags, extra info |
| `created_at` | INT64 | Unix timestamp |

**Index:** HNSW (Hierarchical Navigable Small World) with COSINE metric — optimized for fast approximate nearest-neighbor search.

In [ ]:
from src.vector_db.client import get_vector_client

client = get_vector_client()

print(f'🔗 Connected to Milvus at {settings.milvus_host}:{settings.milvus_port}')
print(f'📦 Collection: "{client.collection_name}"')
print(f'📊 Total entities: {client.count()}')
print()

# List all documents
docs = client.list_documents(limit=10)
print(f'📄 Documents in knowledge base:')
for d in docs:
    print(f'  - {d["source"]} ({d["chunk_count"]} chunks, doc_id={d["doc_id"][:8]}...)')

---

## 🔍 Stage B: Retrieval Pipeline (Query Time)

Now let's trace what happens when a user sends a query.

### B.1 HyDE — Hypothetical Document Embeddings

**Problem:** User queries are short and vague. Documents are long and detailed.
Their embeddings live in different parts of vector space.

**Solution (HyDE):** Ask the LLM to generate a *hypothetical answer* to the query.
Then embed _that answer_ instead of the original query. The hypothetical document is
closer in embedding space to real documents, improving recall.

```
"What is RAG?"  ──LLM──▶  "RAG is a technique that combines..."  ──Embed──▶  Better vector
```

In [ ]:
from src.services.retrieval.hyde import HyDEGenerator

hyde = HyDEGenerator()

query = "How does hybrid search work in RAG systems?"
print(f'📝 Original query: "{query}"')
print()

# HyDE generates a hypothetical document and embeds it
hyde_embedding, hypothetical_doc = hyde.generate_hypothetical_embedding(query)

# Also embed the original query for comparison
original_embedding = embedder.embed_query(query)

print(f'🤖 LLM-generated hypothetical document:')
print(f'   "{hypothetical_doc[:300]}..."')
print()
print(f'📏 Original query embedding dim: {len(original_embedding)}')
print(f'📏 HyDE embedding dim:           {len(hyde_embedding)}')
print(f'📊 Similarity (original ↔ HyDE): {cosine_sim(original_embedding, hyde_embedding):.4f}')
print()
print('→ The HyDE embedding should be closer to actual document embeddings, improving retrieval!')

### B.2 Hybrid Search — Dense + Sparse + RRF Fusion

We don't rely on a single search strategy. Instead, we combine **two complementary approaches**:

| Strategy | Method | What It Captures |
|----------|--------|------------------|
| **Dense** | Milvus HNSW/COSINE | Semantic meaning — "dog" matches "puppy" |
| **Sparse** | BM25 (in-memory) | Exact keywords — "Python 3.12" matches "Python 3.12" |

**Reciprocal Rank Fusion (RRF)** merges the two ranked lists:

```
RRF_score(d) = Σ 1 / (k + rank_i(d))
```

Where `k=60` is a constant that prevents high-ranked items from dominating.

In [ ]:
from src.services.retrieval.hybrid_search import HybridSearcher

searcher = HybridSearcher()

query = "How does RAG reduce hallucinations?"
query_embedding = embedder.embed_query(query)

print(f'🔍 Query: "{query}"')
print()

# Execute hybrid search
results = searcher.search(
    query=query,
    query_embedding=query_embedding,
    top_k=10,     # Retrieve 10 candidates
    final_k=5,    # Return top 5 after fusion
)

print(f'📊 Results: {len(results)} chunks retrieved')
print()
for i, r in enumerate(results):
    rrf = r.get('rrf_score', r.get('score', 0))
    print(f'  {i+1}. [RRF={rrf:.4f}] {r["text"][:120]}...')
    print(f'     Source: {r.get("source", "unknown")}')
    print()

### B.3 Cross-Encoder Reranking

**Why rerank?** Bi-encoder search (Step B.2) is fast but approximate. A cross-encoder
is much more accurate because it processes the (query, passage) pair *together*.

```
Bi-encoder (fast):    embed(query) ↔ embed(passage) → similarity
Cross-encoder (slow): model(query + passage) → relevance score
```

We use `cross-encoder/ms-marco-MiniLM-L-6-v2` — trained on passage ranking data.

In [ ]:
from src.services.retrieval.reranker import get_reranker

if results:
    reranker = get_reranker()

    print(f'📊 Before reranking (top {len(results)} from hybrid search):')
    for i, r in enumerate(results):
        print(f'  {i+1}. {r["text"][:80]}...')

    # Rerank
    reranked = reranker.rerank(query, results, top_k=3)

    print(f'\n📊 After reranking (top {len(reranked)}):')
    for i, r in enumerate(reranked):
        score = r.get('rerank_score', 0)
        print(f'  {i+1}. [score={score:.4f}] {r["text"][:80]}...')

    print('\n→ Notice how the order may change — cross-encoder finds the truly most relevant passages!')
else:
    print('⚠️  No results to rerank. Ingest some documents first!')

---

## 🧩 Stage C: Response Generation

### C.1 Prompt Optimization — Lost-in-the-Middle

Research shows that LLMs pay **more attention to the beginning and end** of the context window,
and tend to "lose" information in the middle ([Liu et al., 2023](https://arxiv.org/abs/2307.03172)).

Our optimizer reorders the chunks:
```
Position:  [1]   [2]   [3]   [4]   [5]
Score:     Best  3rd   5th   4th   2nd    ← Best at edges, weakest in middle
```

It also **compresses** text by removing filler phrases and normalizing whitespace.

In [ ]:
from src.services.generator.prompt_optimizer import PromptOptimizer

optimizer = PromptOptimizer()

if results:
    print(f'📦 Input: {len(results)} chunks')
    total_chars = sum(len(r.get('text', '')) for r in results)
    print(f'📏 Total characters: {total_chars}')

    optimized = optimizer.optimize(results)

    optimized_chars = sum(len(r.get('text', '')) for r in optimized)
    print(f'✂️  After optimization: {optimized_chars} characters ({optimized_chars/total_chars*100:.1f}%)')
    print()

    print('📐 Reordered chunks (best at edges):')
    for i, r in enumerate(optimized):
        score = r.get('rerank_score', r.get('rrf_score', r.get('score', 0)))
        pos = '🟢 START' if i == 0 else ('🟢 END' if i == len(optimized)-1 else '⚪ MID')
        print(f'  {pos} [{score:.3f}] {r["text"][:60]}...')
else:
    print('⚠️  No results to optimize.')

### C.2 LLM Generation

The optimized context + query are sent to the LLM (LM Studio / OpenAI / Ollama).

The system prompt enforces **grounding** — the LLM must only answer from the provided context:

```
"You are a helpful assistant. Answer the question based ONLY on the context below.
 If the answer is not in the context, say so. Do not make things up."
```

In [ ]:
from src.services.generator.llm_client import get_llm_client

llm = get_llm_client()

print(f'🤖 LLM Provider: {settings.llm_provider}')
print(f'📡 Base URL: {settings.llm_base_url}')
print(f'🧠 Model: {settings.llm_model}')
print()

if results:
    answer = llm.generate_with_context(query, optimized)
    print(f'❓ Query: "{query}"')
    print()
    print(f'✅ Answer:')
    print(f'   {answer}')
else:
    print('⚠️  No context available. Ingest documents first!')

### C.3 Self-RAG — Self-Reflection Loop

Self-RAG adds a **quality control step** after generation:

1. The LLM evaluates its own answer for:
   - **Relevance** — does it address the query?
   - **Grounding** — is it supported by the context?
   - **Completeness** — does it fully answer the question?

2. If confidence is below threshold:
   - Generate a refined query
   - Re-retrieve additional context
   - Merge with original context
   - Regenerate the answer

```
Answer ──▶ Evaluate ──▶ Confidence < 0.5? ──▶ Retry with refined query
                              │
                              └── Confidence ≥ 0.5? ──▶ Return answer
```

In [ ]:
from src.services.generator.self_rag import SelfRAG

self_rag = SelfRAG()

if results and 'answer' in dir():
    print(f'🔍 Evaluating answer quality...')
    print()

    evaluation = self_rag.evaluate(query, optimized, answer)

    print(f'📊 Self-RAG Evaluation:')
    print(f'  Confidence:  {evaluation.get("confidence", 0):.2f}')
    print(f'  Is relevant: {evaluation.get("is_relevant", "unknown")}')
    print(f'  Is grounded: {evaluation.get("is_grounded", "unknown")}')
    print(f'  Is complete: {evaluation.get("is_complete", "unknown")}')
    print()

    should_retry, refined_query = self_rag.should_retry(evaluation)
    if should_retry:
        print(f'🔄 Self-RAG says: RETRY with refined query: "{refined_query}"')
    else:
        print(f'✅ Self-RAG says: Answer is good enough!')
else:
    print('⚠️  Need results and answer from previous cells.')

---

## 🎯 Stage D: Full Pipeline (End to End)

The `RAGController` orchestrates all the above steps in a single call.
Let's run the complete pipeline and see the full trace.

In [ ]:
from src.controllers.rag_controller import get_rag_controller

controller = get_rag_controller()

query = "What techniques are used in modern RAG systems?"
print(f'❓ Query: "{query}"')
print('─' * 60)

result = controller.query(
    query=query,
    enable_hyde=True,
    enable_reranking=True,
    enable_self_rag=True,
    top_k=3,
)

print(f'\n✅ Answer:')
print(f'   {result["answer"]}')
print()
print(f'📚 Sources ({len(result["sources"])}):')
for i, src in enumerate(result['sources']):
    print(f'  {i+1}. [{src["score"]:.3f}] {src["source"]} — {src["text"][:80]}...')
print()
print(f'📊 Pipeline Metadata:')
meta = result['metadata']
print(f'  HyDE used:      {meta.get("hyde_used", False)}')
print(f'  Reranking used:  {meta.get("reranking_used", False)}')
print(f'  Self-RAG used:   {meta.get("self_rag_used", False)}')
print(f'  Chunks retrieved: {meta.get("retrieval_count", 0)}')
if meta.get('self_rag_confidence'):
    print(f'  Self-RAG confidence: {meta["self_rag_confidence"]:.2f}')
if meta.get('self_rag_retried'):
    print(f'  Self-RAG retried:    Yes (refined query: "{meta.get("refined_query", "")}")')

---

## 🔑 Stage E: Authentication & API

The API layer wraps everything above in a secure HTTP interface.

### Authentication Flow

```
Register ──▶ Login ──▶ Get JWT ──▶ Use JWT for all requests
```

**JWT Token Contents:**
```json
{
  "sub": "username",
  "role": "user|admin",
  "org_id": "default",
  "user_id": "username",
  "exp": 1771390786
}
```

### Role-Based Access Control

| Endpoint | Required Role |
|----------|---------------|
| Register / Login | None |
| Ingest / Query / Me | Any authenticated user |
| List Users | Admin only |
| Delete Documents | Owner only (can't delete other users' docs) |

In [ ]:
import requests

BASE = 'http://localhost:8081'

# 1. Register
r = requests.post(f'{BASE}/api/v1/auth/register', json={
    'username': 'notebook_user',
    'password': 'test123456',
    'org_id': 'notebooks'
})
print(f'1. Register: {r.status_code} — {r.json()}')

# 2. Login
r = requests.post(f'{BASE}/api/v1/auth/login', json={
    'username': 'notebook_user',
    'password': 'test123456'
})
token = r.json().get('access_token', '')
print(f'2. Login:    {r.status_code} — token={token[:20]}...')

headers = {'Authorization': f'Bearer {token}'}

# 3. Get current user
r = requests.get(f'{BASE}/api/v1/auth/me', headers=headers)
print(f'3. Me:       {r.status_code} — {r.json()}')

# 4. List documents
r = requests.get(f'{BASE}/api/v1/ingest', headers=headers)
print(f'4. Docs:     {r.status_code} — {len(r.json())} documents found')

---

## 📊 Summary — Architecture Recap

```
┌──────────────────────────────────────────────────────────────────────┐
│                         RAG-LLM Architecture                        │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────┐   ┌──────────┐   ┌───────────┐   ┌──────────────────┐ │
│  │ FastAPI  │   │   JWT    │   │   Rate    │   │   Swagger UI     │ │
│  │ Router   │◀─▶│  Auth    │◀─▶│  Limiter  │   │   Auto-generated │ │
│  └────┬─────┘   └──────────┘   └───────────┘   └──────────────────┘ │
│       │                                                              │
│  ┌────▼──────────────────────────────────────────────────────────┐  │
│  │                     RAG Controller                            │  │
│  │  ┌──────┐ ┌────────┐ ┌────────┐ ┌──────────┐ ┌───────────┐  │  │
│  │  │ HyDE │▶│ Hybrid │▶│Reranker│▶│ Prompt   │▶│    LLM    │  │  │
│  │  │      │ │ Search │ │        │ │ Optimize │ │ Generate  │  │  │
│  │  └──────┘ └────────┘ └────────┘ └──────────┘ └─────┬─────┘  │  │
│  │                                                     │        │  │
│  │                                              ┌──────▼──────┐ │  │
│  │                                              │  Self-RAG   │ │  │
│  │                                              │  Reflection │ │  │
│  │                                              └─────────────┘ │  │
│  └───────────────────────────────────────────────────────────────┘  │
│       │                          │                                   │
│  ┌────▼─────┐            ┌───────▼────┐                             │
│  │  Milvus  │            │  LM Studio │                             │
│  │ VectorDB │            │  / OpenAI  │                             │
│  └──────────┘            └────────────┘                             │
└──────────────────────────────────────────────────────────────────────┘
```

### Key Design Decisions

1. **Hybrid Search > Single Strategy** — Dense catches semantics, sparse catches keywords
2. **Cross-encoder Reranking** — Bi-encoder retrieval is fast but imprecise; reranking adds precision
3. **HyDE** — Bridges the query-document gap in embedding space
4. **Lost-in-the-Middle** — Exploits LLM attention patterns for better context utilization
5. **Self-RAG** — Catches bad answers before they reach the user
6. **Feature Toggles** — All advanced features can be enabled/disabled per-request
7. **User Isolation** — Documents and deletions are scoped to the authenticated user